Recompute supplemental publication tables, statistical checks and supporting figures from included data.


In [1]:
import json
import os
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
from scipy import stats
warnings.filterwarnings('ignore')
from pathlib import Path
from types import SimpleNamespace
from IPython.display import display
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def locate_repository():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'data/limfmed_dataset_gold.csv').exists() and (parent / 'experiments/clinical_protocols').is_dir():
            resolved = str(parent.resolve())
            return Path(chr(92) * 2 + chr(63) + chr(92) + resolved) if not resolved.startswith(chr(92) * 2 + chr(63) + chr(92)) else Path(resolved)
    raise FileNotFoundError('Repository root not found')
ROOT = locate_repository()
HERE = str(ROOT / 'analyses/clinical_evaluation')
PAPER = str(ROOT / 'results/clinical_evaluation')
GOLD = str(ROOT)
ALPHA = GOLD
RESEARCH = GOLD
TABLES = str(ROOT / 'results/clinical_evaluation/tables/supplementary')
FIGURES = str(ROOT / 'results/clinical_evaluation/figures/supplementary')
DATASET = str(ROOT / 'data/limfmed_dataset_gold.csv')
Path(TABLES).mkdir(parents=True, exist_ok=True)
Path(FIGURES).mkdir(parents=True, exist_ok=True)
PBASE = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
PBASE_NON = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_truncated'
ORG_DS = 'experiments/clinical_protocols/delphi_panels/heterogeneous_deepseek_organizer_complete'
ORG_QWEN = 'experiments/clinical_protocols/delphi_panels/heterogeneous_qwen_organizer_complete'
ORG_GEM = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gemini_external_organizer'
PWEAK = 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro'
PMIXED = 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm'
PSTRONG = 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'
HOMOG = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'deepseekv4': 'experiments/homogeneous_panels/three_deepseekv4_members_complete', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
FIX_PBASE = 'experiments/fixed_length_controls/delphi_panels/internal_panel_variants/standard_protocol/heterogeneous_gpt120_organizer'
SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
MODEL_NAME = {'gpt': 'gpt-oss-120b', 'qwen': 'qwen3-235b-a22b', 'deepseekv4': 'deepseek-v4-flash', 'deepseekv4pro': 'deepseek-v4-pro', 'gemini': 'gemini-3.7-flash', 'glm': 'glm-5.2', 'kimi': 'kimi'}
PANELS = {'P-base': (PBASE, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'gpt', True), 'P-base/org-ds': (ORG_DS, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'deepseekv4', True), 'P-base/org-qwen': (ORG_QWEN, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'qwen', True), 'P-base/org-gemini': (ORG_GEM, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'gemini', False), 'P-weak': (PWEAK, 'delphi_pweak_{d}_judge.csv', ['qwen', 'gemini', 'deepseekv4pro'], 'qwen', True), 'P-mixed': (PMIXED, 'delphi_pmixed_{d}_judge.csv', ['deepseekv4', 'gemini', 'glm'], 'deepseekv4', True), 'P-strong': (PSTRONG, 'delphi_pstrong_{d}_judge.csv', ['deepseekv4', 'glm', 'gpt'], 'deepseekv4', True), 'homog gpt': (HOMOG['gpt'], 'delphi_homog_gpt_{d}_judge.csv', ['gpt'] * 3, 'gpt', True), 'homog qwen': (HOMOG['qwen'], 'delphi_homog_qwen_{d}_judge.csv', ['qwen'] * 3, 'qwen', True), 'homog deepseekv4': (HOMOG['deepseekv4'], 'delphi_homog_deepseekv4_{d}_judge.csv', ['deepseekv4'] * 3, 'deepseekv4', True), 'homog gemini': (HOMOG['gemini'], 'delphi_homog_gemini_{d}_judge.csv', ['gemini'] * 3, 'gemini', True)}
HETERO = ['P-base', 'P-weak', 'P-mixed', 'P-strong']
HOMOGS = ['homog gpt', 'homog qwen', 'homog deepseekv4', 'homog gemini']
ORGANIZER_ARMS = ['P-base', 'P-base/org-ds', 'P-base/org-qwen', 'P-base/org-gemini']
RAW_RUN = {'P-base': PBASE + '/delphi_noroles.csv', 'P-base/org-ds': ORG_DS + '/delphi_noroles.csv', 'P-base/org-qwen': ORG_QWEN + '/delphi_noroles.csv', 'P-base/org-gemini': ORG_GEM + '/delphi_noroles.csv', 'P-weak': PWEAK + '/delphi_pweak_noroles.csv', 'P-mixed': PMIXED + '/delphi_pmixed_noroles.csv', 'P-strong': PSTRONG + '/delphi_pstrong_noroles.csv', 'homog gpt': HOMOG['gpt'] + '/delphi_homog_gpt_noroles.csv', 'homog qwen': HOMOG['qwen'] + '/delphi_homog_qwen_noroles.csv', 'homog deepseekv4': HOMOG['deepseekv4'] + '/delphi_homog_deepseekv4_noroles.csv', 'homog gemini': HOMOG['gemini'] + '/delphi_homog_gemini_noroles.csv'}
ROUNDS_EVAL = {'P-base': (PBASE, 'delphi_base_noroles_protocol_r{r}_judge.csv'), 'P-weak': (PWEAK, 'delphi_pweak_noroles_protocol_r{r}_judge.csv'), 'P-mixed': (PMIXED, 'delphi_pmixed_noroles_protocol_r{r}_judge.csv'), 'P-strong': (PSTRONG, 'delphi_pstrong_noroles_protocol_r{r}_judge.csv'), 'homog gpt': (HOMOG['gpt'], 'delphi_homog_gpt_noroles_protocol_r{r}_judge.csv'), 'homog gemini': (HOMOG['gemini'], 'delphi_homog_gemini_noroles_protocol_r{r}_judge.csv'), 'homog deepseekv4': (HOMOG['deepseekv4'], 'delphi_homog_deepseekv4_noroles_protocol_r{r}_judge.csv')}

def baseline_path(model, design='noroles', variant='nofix'):
    return f"experiments/{('clinical_protocols' if variant == 'nofix' else 'fixed_length_controls')}/single_model_baselines/baseline_{model}/llm_as_judge/baseline_{design}_{model}_judge.csv"

def panel_path(name, design='noroles'):
    d, f, *_ = PANELS[name]
    return d + '/llm_as_judge/' + f.format(d=design)

def _abs(rel):
    return os.path.normpath(rel if os.path.isabs(rel) else os.path.join(GOLD, rel))

def read_csv(rel, filter_result=True):
    p = _abs(rel)
    if not os.path.exists(p):
        return None
    d = pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False)
    d.columns = [c.lstrip('\ufeff') for c in d.columns]
    if 'Vignette' in d:
        d['Vignette'] = d['Vignette'].astype(str).str.replace(chr(13) + chr(10), chr(10), regex=False).str.replace(chr(13), chr(10), regex=False)
    if filter_result and 'table_source' in d.columns:
        d = d[d['table_source'].fillna('result_df') == 'result_df']
    return d

def read_judge(rel):
    d = read_csv(rel)
    if d is None or 'Judge_score' not in d.columns:
        return None
    d = d.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette', keep='first')
    d['Judge_score'] = pd.to_numeric(d['Judge_score'], errors='coerce')
    return d.set_index('Vignette')

def load_main_arms(design='noroles', include_usc=False):
    arms = {}
    for m in SINGLES:
        d = read_judge(baseline_path(m, design))
        if d is not None:
            arms[m] = d
    for name in PANELS:
        d = read_judge(panel_path(name, design))
        if d is not None:
            arms[name] = d
    if include_usc and design == 'noroles':
        for k in (3, 5, 7):
            d = read_judge(f'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc{k}_judge.csv')
            if d is not None:
                arms[f'USC@{k} gpt'] = d
    return arms

def score(arms, name):
    return arms[name]['Judge_score'].astype(float)

def dataset():
    d = pd.read_csv(DATASET, sep=None, engine='python')
    return d

def protocol_text(d):
    for c in ('winner_protocol', 'baseline_protocol_anonymous', 'baseline_protocol'):
        if c in d.columns:
            return d[c].astype(str)
    raise KeyError('no protocol column')

def strip_author(text):
    if not isinstance(text, str):
        return ''
    return re.split('(?im)^\\s*Protocol Author\\s*:', text, maxsplit=1)[0].rstrip()

def n_words(text):
    return len(re.findall('\\S+', strip_author(text))) if isinstance(text, str) else np.nan

def items(d):
    rows = []
    for v, s in d['judge_details'].dropna().items():
        try:
            arr = json.loads(s)
        except Exception:
            continue
        for it in arr:
            ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
            rows.append((v, int(it['id']), int(it.get('points', 0) or 0), ax, bool(it.get('met'))))
    return pd.DataFrame(rows, columns=['Vignette', 'cid', 'points', 'axis', 'met'])

def item_matrix(arms, names, positive_only=True):
    frames = []
    for n in names:
        it = items(arms[n]).set_index(['Vignette', 'cid'])
        frames.append(it['met'].astype(int).rename(n))
        meta = it[['points', 'axis']]
    m = pd.concat(frames, axis=1, join='inner')
    m = m.join(meta, how='left')
    if positive_only:
        m = m[m['points'] > 0]
    return m
ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
_DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')

def norm_label(x):
    if not isinstance(x, str) or not x.strip():
        return None
    x = x.strip()
    if x in GT:
        return GT[x]
    s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
    s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
    for k, v in (('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')):
        if k in s:
            return v
    if 'non-emergent' in s or 'non emergent' in s:
        return 'NONEMERG'
    if 'emergent' in s or 'emergency' in s:
        return 'EMERG'
    if 'self' in s:
        return 'SELF'
    return None
_PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

def parse_triage(text):
    if not isinstance(text, str) or not text.strip():
        return None
    m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
    return norm_label(m.group(1)) if m else None

def predicted_triage(d):
    if 'delphi_consensus_emergency_type' in d.columns:
        s = d['delphi_consensus_emergency_type'].map(norm_label)
        if s.notna().mean() > 0.5:
            return s
    return protocol_text(d).map(parse_triage)

def true_triage(d):
    return d['Type'].map(GT)

def participants(d):
    return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

def paired(a, b):
    ix = a.dropna().index.intersection(b.dropna().index)
    x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
    dd = x - y
    p = stats.wilcoxon(x, y).pvalue if (dd != 0).any() else np.nan
    return {'n': len(ix), 'delta': float(dd.mean()), 'share_gt': float((dd > 0).mean()), 'p_wilcoxon': float(p)}

def boot_ci(x, B=10000, seed=20260922, alpha=0.05, stat=np.mean):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float)
    n = len(x)
    idx = rng.integers(0, n, (B, n))
    s = stat(x[idx], axis=1)
    return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

def boot_ci_two_level(x, cluster, B=10000, seed=20260922, alpha=0.05):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
    groups = [g['x'].to_numpy() for _, g in df.groupby('c')]
    G = len(groups)
    out = np.empty(B)
    for b in range(B):
        pick = rng.integers(0, G, G)
        vals = []
        for i in pick:
            g = groups[i]
            vals.append(g[rng.integers(0, len(g), len(g))])
        out[b] = np.concatenate(vals).mean()
    return (float(np.quantile(out, alpha / 2)), float(np.quantile(out, 1 - alpha / 2)))

def holm(p):
    p = np.asarray(p, float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for i, k in enumerate(order):
        run = max(run, (m - i) * p[k])
        adj[k] = min(run, 1.0)
    return adj

def mcnemar_exact(a, b):
    ix = a.dropna().index.intersection(b.dropna().index)
    x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
    n10, n01 = (int((x & ~y).sum()), int((~x & y).sum()))
    p = stats.binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    return {'n': len(ix), 'acc_a': float(x.mean()), 'acc_b': float(y.mean()), 'n10': n10, 'n01': n01, 'p': float(p)}

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return (c - h, c + h)

def tost_paired(dvec, margin):
    d = np.asarray(dvec, float)
    n = len(d)
    m, se = (d.mean(), d.std(ddof=1) / np.sqrt(n))
    p_low = stats.t.sf((m + margin) / se, n - 1)
    p_up = stats.t.cdf((m - margin) / se, n - 1)
    return float(max(p_low, p_up))

def fmt_p(p):
    if p is None or not np.isfinite(p):
        return 'n/a'
    if p < 0.0001:
        return f'{p:.1e}'
    return f'{p:.3f}' if p >= 0.001 else f'{p:.4f}'

def save_table(df, name, index=True):
    path = os.path.join(TABLES, name)
    df.to_csv(path, sep=';', encoding='utf-8-sig', index=index)
    print(f'saved tables/{name} ({len(df)} rows)')
    return path


In [2]:
P = SimpleNamespace(**{key: value for key, value in globals().copy().items() if not key.startswith('_')})
arms = load_main_arms()
NAMES = {'P-base': 'D-1G', 'P-base/org-ds': 'D-1D', 'P-base/org-qwen': 'D-1Q', 'P-base/org-gemini': 'D-1Ge', 'P-weak': 'D-3Q', 'P-mixed': 'D-4D', 'P-strong': 'D-2D', 'homog gpt': 'DH-G', 'homog qwen': 'DH-Q', 'homog deepseekv4': 'DH-D', 'homog gemini': 'DH-Ge', **MODEL_NAME}
exports = {}

def export(frame, name):
    frame = frame.copy()
    frame.to_csv(Path(TABLES) / (name + '.csv'), index=False, encoding='utf-8-sig')
    exports[name] = frame
    return frame

def coefficient(fit, term, **extra):
    ci = fit.conf_int().loc[term]
    return {**extra, 'term': term, 'observations': int(fit.nobs), 'estimate': float(fit.params[term]), 'ci_low': float(ci.iloc[0]), 'ci_high': float(ci.iloc[1]), 'p': float(fit.pvalues[term])}
display(pd.DataFrame({'system': list(arms), 'scored_protocols': [len(d) for d in arms.values()]}))


,system,scored_protocols
0,gpt,229
1,qwen,229
2,deepseekv4,228
3,deepseekv4pro,229
4,gemini,229
5,glm,229
6,kimi,221
7,P-base,229
8,P-base/org-ds,229
9,P-base/org-qwen,229


In [3]:
matrix_rows = []
for key in [*SINGLES, *PANELS]:
    for design in ['noroles', 'randomroles']:
        if key in SINGLES:
            full = read_judge(baseline_path(key, design))
            fixed = read_judge(baseline_path(key, design, 'fix'))
            short = read_judge(f'experiments/clinical_protocols/single_model_baselines/baseline_{key}/llm_as_judge/judge_robustness/baseline_{design}_{key}_short_judge.csv')
        else:
            full = read_judge(panel_path(key, design))
            fixed = read_judge(FIX_PBASE + f'/llm_as_judge/delphi_base_{design}_judge.csv') if key == 'P-base' else None
            short = read_judge(PBASE + f'/llm_as_judge/judge_robustness/delphi_base_{design}_short_judge.csv') if key == 'P-base' else None
        for regime, data in [('original', full), ('short_vignette', short), ('400_500_words', fixed)]:
            if data is not None:
                matrix_rows.append({'key': key, 'system': NAMES[key], 'regime': regime, 'design': design, 'n': len(data), 'mean_score': data.Judge_score.mean()})
matrix = pd.DataFrame(matrix_rows)
export(matrix[matrix.key.isin(SINGLES)], 'baselines_matrix')
export(matrix[matrix.key.isin(PANELS)], 'panels_matrix')
distribution = []
for key, d in arms.items():
    y = d.Judge_score.astype(float)
    it = items(d)
    npen = it[it.points.lt(0) & it.met].Vignette.nunique()
    distribution.append({'key': key, 'system': NAMES[key], 'n': len(y), 'mean': y.mean(), 'sd': y.std(ddof=1), 'below_0_5': int(y.lt(0.5).sum()), 'below_0_5_pct': 100 * y.lt(0.5).mean(), 'any_penalty': npen, 'any_penalty_pct': 100 * npen / len(y), 'q10': y.quantile(0.1), 'median': y.median()})
export(pd.DataFrame(distribution), 'score_distribution')


,key,system,n,mean,sd,below_0_5,below_0_5_pct,any_penalty,any_penalty_pct,q10,median
0,gpt,gpt-oss-120b,229,0.625811,0.171362,52,22.707424,54,23.580786,0.41430,0.6308
1,qwen,qwen3-235b-a22b,229,0.562228,0.173540,77,33.624454,56,24.454148,0.33142,0.5846
2,deepseekv4,deepseek-v4-flash,228,0.640056,0.164053,50,21.929825,40,17.543860,0.41156,0.6593
3,deepseekv4pro,deepseek-v4-pro,229,0.560091,0.189813,81,35.371179,52,22.707424,0.30268,0.5938
4,gemini,gemini-3.7-flash,229,0.424720,0.187322,150,65.502183,84,36.681223,0.18236,0.4364
5,glm,glm-5.2,229,0.587015,0.192234,75,32.751092,54,23.580786,0.34720,0.6000
6,kimi,kimi,221,0.595483,0.192714,68,30.769231,46,20.814480,0.34570,0.6269
7,P-base,D-1G,229,0.789946,0.127825,2,0.873362,12,5.240175,0.60392,0.8000
8,P-base/org-ds,D-1D,229,0.811066,0.122176,3,1.310044,13,5.676856,0.62794,0.8364
9,P-base/org-qwen,D-1Q,229,0.759186,0.140222,6,2.620087,17,7.423581,0.57140,0.7794


In [4]:
long = []
for key, d in arms.items():
    f = d[['Judge_score', 'Genealogy', 'Type', 'Age Group', 'Sub Clinical Specialties']].copy()
    f['case'] = d.index
    f['system'] = key
    long.append(f.rename(columns={'Judge_score': 'score', 'Genealogy': 'source', 'Type': 'urgency', 'Age Group': 'age', 'Sub Clinical Specialties': 'specialty'}))
clinical = pd.concat(long, ignore_index=True)
variance = []
for factor in ['source', 'urgency', 'specialty', 'age', 'system', 'case']:
    m = smf.ols('score ~ C(' + factor + ')', clinical).fit()
    variance.append({'factor': factor, 'n': int(m.nobs), 'levels': clinical[factor].nunique(), 'r_squared': m.rsquared, 'adjusted_r_squared': m.rsquared_adj})
export(pd.DataFrame(variance), 'variance')
source_summary = clinical.groupby('source', sort=True).agg(cases=('case', 'nunique'), scored_protocols=('score', 'size'), mean_score=('score', 'mean')).reset_index()
export(source_summary, 'source_difficulty')
sf = smf.ols('score ~ C(system) + C(source)', clinical).fit(cov_type='cluster', cov_kwds={'groups': clinical.case}, use_t=False)
export(pd.DataFrame([coefficient(sf, t, cases=clinical.case.nunique()) for t in sf.params.index if t.startswith('C(source)')]), 'source_difficulty_model')
member = pd.concat([arms[m].Judge_score.rename(m) for m in ['gpt', 'qwen', 'deepseekv4']], axis=1).dropna()
gap = (arms['P-base'].Judge_score.reindex(member.index) - member.max(axis=1)).rename('gain')
subgroup = []
subgroup_tests = []
for factor in ['Genealogy', 'Type', 'Age Group', 'Sub Clinical Specialties']:
    grouped = list(gap.groupby(arms['P-base'][factor].reindex(gap.index)))
    eligible = [g for _, g in grouped if len(g) >= 5]
    test = stats.kruskal(*eligible)
    subgroup_tests.append({'factor': factor, 'groups': len(eligible), 'cases': sum(map(len, eligible)), 'H': test.statistic, 'p': test.pvalue})
    for level, g in grouped:
        ci = boot_ci(g.to_numpy())
        subgroup.append({'factor': factor, 'level': level, 'n': len(g), 'gain': g.mean(), 'ci_low': ci[0], 'ci_high': ci[1], 'p': stats.wilcoxon(g).pvalue if np.any(g != 0) else 1.0})
export(pd.DataFrame(subgroup_tests), 'subgroup_tests')
export(pd.DataFrame(subgroup), 'subgroups')


,factor,level,n,gain,ci_low,ci_high,p
0,Genealogy,AI,151,0.087468,0.069783,0.105482,7.821536e-15
1,Genealogy,checker,43,0.103451,0.065402,0.141607,3.881736e-05
2,Genealogy,msd,34,0.072771,0.034529,0.110497,1.430283e-03
3,Type,Requires emergent care,84,0.084539,0.062121,0.107293,3.721034e-09
4,Type,Requires non-emergent care,80,0.099615,0.073392,0.125724,7.529011e-09
5,Type,Self-care appropriate,64,0.079058,0.051111,0.107605,7.709689e-06
6,Age Group,age not specified,1,0.160500,0.160500,0.160500,1.000000e+00
7,Age Group,children (<15),35,0.097069,0.055017,0.138643,3.610925e-04
8,Age Group,elderly age (>60),37,0.074681,0.035110,0.116721,4.773854e-03
9,Age Group,middle age (45-60),44,0.091550,0.060370,0.121444,1.005706e-05


In [5]:
fixed = {m: read_judge(baseline_path(m, 'noroles', 'fix')) for m in ['gpt', 'qwen', 'deepseekv4']}
fixed['P-base'] = read_judge(FIX_PBASE + '/llm_as_judge/delphi_base_noroles_judge.csv')

def length_frame(available, panel, comparators):
    keys = [panel, *comparators]
    indices = available[keys[0]].index
    for key in keys[1:]:
        indices = indices.intersection(available[key].index)
    frames = []
    for key in keys:
        data = available[key].loc[indices]
        frames.append(pd.DataFrame({'case': indices, 'system': key, 'y': data.Judge_score.to_numpy(), 'panel': int(key == panel), 'lw': np.log(protocol_text(data).map(n_words).to_numpy())}))
    return pd.concat(frames, ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
length_rows = []
contrasts = [('P-base', [m]) for m in ['gpt', 'qwen', 'deepseekv4']] + [('P-base', ['gpt', 'qwen', 'deepseekv4'])]
for regime, available in [('original', arms), ('400_500_words', fixed)]:
    for panel, comparators in contrasts:
        data = length_frame(available, panel, comparators)
        before = smf.ols('y ~ panel + C(case)', data).fit(cov_type='cluster', cov_kwds={'groups': data.case}, use_t=True)
        after = smf.ols('y ~ panel + lw + C(case)', data).fit(cov_type='cluster', cov_kwds={'groups': data.case}, use_t=True)
        row = coefficient(after, 'panel', regime=regime, panel=NAMES[panel], comparator=';'.join((NAMES[k] for k in comparators)), cases=data.case.nunique())
        row.update(unadjusted=before.params['panel'], log_words=after.params['lw'], log_words_ci_low=after.conf_int().loc['lw'].iloc[0], log_words_ci_high=after.conf_int().loc['lw'].iloc[1], attenuation=1 - after.params['panel'] / before.params['panel'])
        length_rows.append(row)
for panel in PANELS:
    if panel == 'P-base':
        continue
    comparator = max(set(PANELS[panel][2]), key=lambda k: arms[k].Judge_score.mean())
    data = length_frame(arms, panel, [comparator])
    before = smf.ols('y ~ panel + C(case)', data).fit(cov_type='cluster', cov_kwds={'groups': data.case}, use_t=True)
    after = smf.ols('y ~ panel + lw + C(case)', data).fit(cov_type='cluster', cov_kwds={'groups': data.case}, use_t=True)
    row = coefficient(after, 'panel', regime='original', panel=NAMES[panel], comparator=NAMES[comparator], cases=data.case.nunique())
    row.update(unadjusted=before.params['panel'], log_words=after.params['lw'], log_words_ci_low=after.conf_int().loc['lw'].iloc[0], log_words_ci_high=after.conf_int().loc['lw'].iloc[1], attenuation=1 - after.params['panel'] / before.params['panel'])
    length_rows.append(row)
export(pd.DataFrame(length_rows), 'length_regression')
data = pd.concat([pd.DataFrame({'case': d.index, 'system': key, 'y': d.Judge_score.to_numpy(), 'lw': np.log(protocol_text(d).map(n_words).to_numpy())}) for key, d in arms.items() if key in ORGANIZER_ARMS], ignore_index=True)
formula = 'y ~ C(system, Treatment(reference="P-base/org-ds")) + C(case)'
before = smf.ols(formula, data).fit(cov_type='cluster', cov_kwds={'groups': data.case}, use_t=True)
after = smf.ols(formula + ' + lw', data).fit(cov_type='cluster', cov_kwds={'groups': data.case}, use_t=True)
organizer = []
for term in after.params.index:
    if term.startswith('C(system'):
        organizer.append(coefficient(after, term, unadjusted=before.params[term], p_unadjusted=before.pvalues[term]))
export(pd.DataFrame(organizer), 'organizer_regression')


,unadjusted,p_unadjusted,term,observations,estimate,ci_low,ci_high,p
0,-0.021120,1.955032e-02,"C(system, Treatment(reference=""P-base/org-ds"")...",916,0.012962,-0.012843,0.038767,0.323339
1,-0.011535,2.727037e-01,"C(system, Treatment(reference=""P-base/org-ds"")...",916,0.008659,-0.012191,0.029508,0.414038
2,-0.051881,3.335437e-08,"C(system, Treatment(reference=""P-base/org-ds"")...",916,-0.015449,-0.042361,0.011463,0.259186


In [6]:
RET = {'P-base': PBASE + '/llm_as_judge/judge_robustness/delphi_base_noroles_retest_r{r}_judge.csv', **{m: f'experiments/clinical_protocols/single_model_baselines/baseline_{m}/llm_as_judge/judge_robustness/baseline_noroles_{m}_retest_r{{r}}_judge.csv' for m in ['gpt', 'qwen', 'deepseekv4']}}
repeated = []
for key, pattern in RET.items():
    for run in range(1, 6):
        data = read_judge(pattern.format(r=run))
        repeated.append(pd.DataFrame({'system': key, 'run': run, 'case': data.index, 'y': data.Judge_score.to_numpy(), 'lw': np.log(protocol_text(data).map(n_words).to_numpy())}))
repeated = pd.concat(repeated, ignore_index=True)
repeated['y_c'] = repeated.y - repeated.groupby(['system', 'case']).y.transform('mean')
repeated['lw_c'] = repeated.lw - repeated.groupby(['system', 'case']).lw.transform('mean')
within = smf.ols('y_c ~ lw_c - 1', repeated).fit(cov_type='cluster', cov_kwds={'groups': repeated.case}, use_t=True)
retest = [coefficient(within, 'lw_c', quantity='Within system-case log-length slope', cases=repeated.case.nunique())]
for key, g in repeated.groupby('system'):
    fit = smf.ols('y ~ C(case) + C(run)', g).fit()
    anova = sm.stats.anova_lm(fit, typ=2)
    retest.append({'quantity': NAMES[key] + ' run effect', 'observations': len(g), 'cases': g.case.nunique(), 'estimate': anova.loc['C(run)', 'F'], 'p': anova.loc['C(run)', 'PR(>F)']})
export(pd.DataFrame(retest), 'retest_length')
export(repeated[['system', 'run', 'case', 'y', 'lw']], 'retest_observations')


,system,run,case,y,lw
0,P-base,1,A 21-year-old man comes to the emergency depar...,0.8235,6.278521
1,P-base,1,A 62-year-old woman with a history of osteoart...,0.8333,5.963579
2,P-base,1,A 34-year-old man presents to urgent care afte...,0.8806,6.232448
3,P-base,1,A 40-year-old man presents with difficulty fal...,0.7313,6.440947
4,P-base,1,A 5-year-old girl is brought to the clinic by ...,0.8235,6.089045
...,...,...,...,...,...
1195,deepseekv4,5,A 31-year-old woman comes to the office becaus...,0.8243,5.855072
1196,deepseekv4,5,A 44-year-old woman with known Addison disease...,0.8750,6.169611
1197,deepseekv4,5,A 55-year-old man with poorly controlled type ...,0.8684,5.993961
1198,deepseekv4,5,A 20-year-old male college athlete presents wi...,0.6562,5.575949


In [7]:
from collections import Counter
ORGANIZER_ID = {'P-base': 'gpt120b_default', 'P-base/org-ds': 'deepseekv4_default', 'P-base/org-qwen': 'qwen_default', 'P-base/org-gemini': None, 'P-weak': 'qwen_default', 'P-mixed': 'deepseekv4_default', 'P-strong': 'deepseekv4_default', 'homog gpt': 'gpt120b_e1', 'homog qwen': 'qwen_e1', 'homog deepseekv4': 'deepseekv4_e1', 'homog gemini': 'gemini_e1'}
PANELS8 = ['P-base', 'P-weak', 'P-mixed', 'P-strong', 'homog gpt', 'homog qwen', 'homog deepseekv4', 'homog gemini']
ALL11 = PANELS8 + ['P-base/org-ds', 'P-base/org-qwen', 'P-base/org-gemini']
SPEC_ALLOWED = {'Cardiology', 'Dermatology', 'Emergency', 'Endocrinology', 'Gastroenterology', 'General Medicine', 'General Surgery', 'Infectious Disease', 'Nephrology', 'Neurology', 'Obstetrics and Gynecology', 'Ophthalmology', 'Orthopedics', 'Other departments', 'Otolaryngology (ENT)', 'Pediatrics', 'Psychiatry', 'Pulmonology', 'Urology'}

def load_run(name):
    d = P.read_csv(P.RAW_RUN[name])
    d = d.dropna(subset=['Vignette']).drop_duplicates('Vignette').set_index('Vignette')
    return d
runs = {n: load_run(n) for n in ALL11}
print({n: (len(d), d.delphi_rounds_executed.value_counts().sort_index().to_dict()) for n, d in runs.items()})

def plurality(labels, org_label=None, confs=None):
    lab = [l for l in labels if l]
    if not lab:
        return None
    c = Counter(lab)
    top = max(c.values())
    tied = [k for k, v in c.items() if v == top]
    if len(tied) == 1:
        return tied[0]
    if org_label in tied:
        return org_label
    if confs is not None:
        s = {k: 0.0 for k in tied}
        for l, cf in zip(labels, confs):
            if l in s and pd.notna(cf):
                s[l] += float(cf)
        return max(tied, key=lambda k: s[k])
    return None

def last_round_col(d, r, field, pid):
    return d.get(f'r{int(r)}_{field}__{pid}')

def long_participants(name, field='emergency', truth=None, norm=P.norm_label):
    d = runs[name]
    pids = P.participants(d)
    org = ORGANIZER_ID[name]
    rows = []
    for v, r in d.iterrows():
        R = int(r['delphi_rounds_executed'])
        t = truth(r)
        r1 = [norm(r.get(f'r1_{field}__{p}')) for p in pids]
        conf1 = [r.get(f'r1_confidence__{p}') for p in pids]
        maj1 = plurality(r1, norm(r.get(f'r1_{field}__{org}')) if org else None, conf1)
        cnt = Counter([x for x in r1 if x])
        for p, l1 in zip(pids, r1):
            lf = norm(r.get(f'r{R}_{field}__{p}')) if R > 1 else l1
            rows.append({'panel': name, 'case': v, 'pid': p, 'rounds': R, 'truth': t, 'r1': l1, 'last': lf, 'maj_r1': maj1, 'n_same_r1': cnt.get(l1, 0) if l1 else 0, 'n_labels_r1': len(cnt)})
    return pd.DataFrame(rows)
truth_type = lambda r: P.GT.get(r['Type'])
truth_spec = lambda r: r['Sub Clinical Specialties'] if r['Sub Clinical Specialties'] in SPEC_ALLOWED else None
norm_spec = lambda x: x.strip() if isinstance(x, str) and x.strip() in SPEC_ALLOWED else None


{'P-base': (229, {1.0: 120, 2.0: 62, 3.0: 47}), 'P-weak': (229, {1.0: 119, 2.0: 61, 3.0: 49}), 'P-mixed': (229, {1.0: 142, 2.0: 44, 3.0: 43}), 'P-strong': (229, {1.0: 128, 2.0: 59, 3.0: 42}), 'homog gpt': (229, {1.0: 173, 2.0: 40, 3.0: 16}), 'homog qwen': (229, {1.0: 197, 2.0: 29, 3.0: 3}), 'homog deepseekv4': (229, {1.0: 171, 2.0: 45, 3.0: 13}), 'homog gemini': (229, {1.0: 192, 2.0: 27, 3.0: 10}), 'P-base/org-ds': (229, {1.0: 117, 2.0: 73, 3.0: 39}), 'P-base/org-qwen': (229, {1.0: 116, 2.0: 76, 3.0: 37}), 'P-base/org-gemini': (229, {1.0: 119, 2.0: 83, 3.0: 27})}


In [8]:
LP = pd.concat([long_participants(n, 'emergency', truth_type) for n in PANELS8], ignore_index=True)
LPc = LP[(LP.rounds >= 2) & LP.truth.notna() & LP.r1.notna() & LP['last'].notna()].copy()
LPc['ok1'] = (LPc.r1 == LPc.truth).astype(int)
LPc['okL'] = (LPc['last'] == LPc.truth).astype(int)
valid3 = LPc.groupby(['panel', 'case']).r1.transform('count') == 3
influence = LPc[valid3 & (LPc.n_labels_r1 == 2)].copy()
influence['position'] = np.where(influence.n_same_r1 == 2, 'majority', 'minority')
majority_label = influence[influence.n_same_r1 == 2].groupby(['panel', 'case']).r1.first().rename('maj')
influence = influence.join(majority_label, on=['panel', 'case']).dropna(subset=['maj'])
influence['maj_correct'] = (influence.maj == influence.truth).astype(int)
influence['ends_on_majority'] = (influence['last'] == influence.maj).astype(int)
fit = smf.logit('ends_on_majority ~ C(position) * maj_correct', influence).fit(disp=0, cov_type='cluster', cov_kwds={'groups': influence.case})
clustered = [coefficient(fit, t, analysis='Majority adherence', cases=influence.case.nunique()) for t in fit.params.index if t != 'Intercept']
LPc['difference'] = LPc.okL - LPc.ok1
individual = smf.ols('difference ~ 1', LPc).fit(cov_type='cluster', cov_kwds={'groups': LPc.case}, use_t=True)
clustered.append(coefficient(individual, 'Intercept', analysis='Individual triage', cases=LPc.case.nunique()))
protocol_changes = []
for key, (directory, pattern) in ROUNDS_EVAL.items():
    initial = read_judge(directory + '/llm_as_judge/round_evaluation/' + pattern.format(r=1))
    final = arms[key]
    rounds = runs[key].delphi_rounds_executed.astype(int)
    ix = initial.index.intersection(final.index).intersection(rounds[rounds >= 2].index)
    protocol_changes.append(pd.DataFrame({'panel': key, 'case': ix, 'difference': (final.Judge_score.loc[ix] - initial.Judge_score.loc[ix]).to_numpy()}))
changes = pd.concat(protocol_changes, ignore_index=True)
protocol = smf.ols('difference ~ 1', changes).fit(cov_type='cluster', cov_kwds={'groups': changes.case}, use_t=True)
clustered.append(coefficient(protocol, 'Intercept', analysis='Protocol synthesis', cases=changes.case.nunique()))
export(pd.DataFrame(clustered), 'rounds_clustered')
export(influence, 'influence_observations')


,panel,case,pid,rounds,truth,r1,last,maj_r1,n_same_r1,n_labels_r1,ok1,okL,position,maj,maj_correct,ends_on_majority
0,P-base,A 21-year-old man comes to the emergency depar...,deepseekv4_default,2,EMERG,NONEMERG,NONEMERG,NONEMERG,2,2,0,0,majority,NONEMERG,0,1
1,P-base,A 21-year-old man comes to the emergency depar...,gpt120b_default,2,EMERG,NONEMERG,NONEMERG,NONEMERG,2,2,0,0,majority,NONEMERG,0,1
2,P-base,A 21-year-old man comes to the emergency depar...,qwen_default,2,EMERG,EMERG,NONEMERG,NONEMERG,1,2,1,0,minority,NONEMERG,0,1
6,P-base,A 34-year-old man presents to urgent care afte...,deepseekv4_default,2,NONEMERG,NONEMERG,NONEMERG,NONEMERG,2,2,1,1,majority,NONEMERG,1,1
7,P-base,A 34-year-old man presents to urgent care afte...,gpt120b_default,2,NONEMERG,NONEMERG,NONEMERG,NONEMERG,2,2,1,1,majority,NONEMERG,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5410,homog gemini,A 45-year-old man presents with acute onset of...,gemini_e2,2,NONEMERG,NONEMERG,NONEMERG,NONEMERG,2,2,1,1,majority,NONEMERG,1,1
5411,homog gemini,A 45-year-old man presents with acute onset of...,gemini_e3,2,NONEMERG,EMERG,NONEMERG,NONEMERG,1,2,0,1,minority,NONEMERG,1,1
5415,homog gemini,A 30-year-old man presents with a 2-day histor...,gemini_e1,2,SELF,SELF,SELF,SELF,2,2,1,1,majority,SELF,1,1
5416,homog gemini,A 30-year-old man presents with a 2-day histor...,gemini_e2,2,SELF,SELF,SELF,SELF,2,2,1,1,majority,SELF,1,1


In [9]:
BASE_TABLES = ROOT / 'results/clinical_evaluation/tables'

def previous(name):
    return pd.read_csv(BASE_TABLES / name, sep=';', encoding='utf-8-sig')
for source, target in [('panel_size_and_quality.csv', 'panel_size60'), ('panel_synthesis_vs_aggregation_rules.csv', 'synthesis_controls'), ('triage_cost_indifference_thresholds.csv', 'triage_tradeoff'), ('medmcqa_accuracy_1000_cases.csv', 'medmcqa_train'), ('clinical_rubric_axes.csv', 'clinical_corpus_rubric_axis_summary'), ('principal_clinical_comparisons.csv', 'principal_system_comparisons')]:
    export(previous(source), target)
med = ROOT / 'experiments/external_benchmarks/medmcqa/benchmark_subsets/expanded_panel_analysis'
for source, target in [('comparisons_accuracy.csv', 'medmcqa_external'), ('rounds_rounds.csv', 'medmcqa_round_gain')]:
    export(pd.read_csv(med / source), target)
comparison = pd.read_csv(med / 'comparisons_panel_vs_baselines.csv')
is_homogeneous = comparison.panel.str.contains('_trio')
export(comparison[~is_homogeneous], 'medmcqa_comparisons')
export(comparison[is_homogeneous], 'medmcqa_homogeneous_comparisons')
human = ROOT / 'evaluations/human_and_model_validation/expert_agreement'
for source, target in [('protocol_overall.csv', 'human_performance'), ('human_pairs.csv', 'human_agreement'), ('protocol_axis_means.csv', 'human_axes')]:
    export(pd.read_csv(human / source), target)
human_results = json.loads((human / 'protocol_comparison_results.json').read_text(encoding='utf-8'))
case_human = pd.read_csv(human / 'protocol_case_means.csv').set_index('case_id') / 100
score_agreement = []
blind_means = []

def icc_absolute_single(a, b):
    values = np.column_stack([a, b])
    n, k = values.shape
    grand = values.mean()
    row = values.mean(axis=1)
    column = values.mean(axis=0)
    msr = k * np.sum((row - grand) ** 2) / (n - 1)
    msc = n * np.sum((column - grand) ** 2) / (k - 1)
    residual = values - row[:, None] - column[None, :] + grand
    mse = np.sum(residual ** 2) / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)
for judge, relative in [('GPT-4.1', 'gpt41_judging/judge_gpt41/blind_protocols_60_gpt41_judge.csv'), ('GPT-4.1 mini', 'gpt41_mini_judging/blind_protocols_60_gpt41_mini_judge.csv')]:
    observed = pd.read_csv(ROOT / 'evaluations/human_and_model_validation' / relative, sep=';', encoding='utf-8-sig')
    ratings = observed.pivot(index='case_id', columns='protocol_source', values='Judge_score').reindex(case_human.index)
    for source in ['all_systems', *case_human.columns]:
        selected = list(case_human.columns) if source == 'all_systems' else [source]
        a = ratings[selected].to_numpy().ravel()
        b = case_human[selected].to_numpy().ravel()
        differences = (ratings[selected] - case_human[selected]).mean(axis=1)
        ci = boot_ci(differences.to_numpy())
        score_agreement.append({'judge': judge, 'source': source, 'n': len(a), 'pearson_r': stats.pearsonr(a, b).statistic, 'spearman_rho': stats.spearmanr(a, b).statistic, 'icc_2_1': icc_absolute_single(a, b), 'llm_minus_human': float((a - b).mean()), 'ci_low': ci[0], 'ci_high': ci[1], 'mean_absolute_difference': float(np.abs(a - b).mean())})
    for source in ratings:
        ci = boot_ci(ratings[source].dropna().to_numpy())
        blind_means.append({'judge': judge, 'source': source, 'n': int(ratings[source].count()), 'mean': ratings[source].mean(), 'ci_low': ci[0], 'ci_high': ci[1]})
export(pd.DataFrame(score_agreement), 'human_scores')
export(pd.DataFrame(blind_means), 'blind_llm_scores')
sensitivity = []
for group in ['sensitivity_common_cases', 'sensitivity_missing_bounds']:
    for row in human_results[group]:
        sensitivity.append({'analysis': group, **row})
for key, value in human_results['sensitivity_equal_expert_weight'].items():
    sensitivity.append({'analysis': 'equal_expert_weight', 'source': key, 'mean': value})
export(pd.DataFrame(sensitivity), 'human_sensitivity')

def save_plot(fig, name):
    for axis in fig.axes:
        if axis.get_legend() is not None:
            axis.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False, title=None)
    fig.tight_layout()
    fig.savefig(Path(FIGURES) / (name + '.pdf'), bbox_inches='tight', metadata={'Creator': '', 'CreationDate': None, 'ModDate': None})
    fig.savefig(Path(FIGURES) / (name + '.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
for factor, name in [('Genealogy', 'source'), ('Type', 'urgency'), ('Age Group', 'age'), ('Sub Clinical Specialties', 'specialty')]:
    data = exports['subgroups']
    data = data[data.factor.eq(factor) & data.n.ge(5)].sort_values('gain')
    fig, ax = plt.subplots(figsize=(7, max(2.8, 0.3 * len(data))))
    ax.errorbar(data.gain, np.arange(len(data)), xerr=[data.gain - data.ci_low, data.ci_high - data.gain], fmt='o', color='#2a78d6')
    ax.set_yticks(np.arange(len(data)), [str(a) + ' (n=' + str(n) + ')' for a, n in zip(data.level, data.n)])
    ax.axvline(0, color='grey', lw=0.8)
    ax.set_xlabel('D-1G minus strongest member for each case')
    save_plot(fig, 'subgroup_' + name)
axis_tables = {}
for key in ['P-base', 'gpt', 'qwen', 'deepseekv4']:
    it = items(arms[key])
    it['net'] = it.points * it.met
    it['maximum'] = it.points.clip(lower=0)
    totals = it.groupby(['Vignette', 'axis']).agg(net=('net', 'sum'), maximum=('maximum', 'sum'))
    totals['score'] = totals.net / totals.maximum.where(totals.maximum.gt(0))
    axis_tables[key] = totals.score.unstack('axis')
axis_rows = []
for axis in axis_tables['P-base'].columns:
    scores = pd.concat([data[axis].rename(key) for key, data in axis_tables.items()], axis=1, join='inner').dropna()
    for key in scores:
        ci = boot_ci(scores[key].to_numpy())
        axis_rows.append({'axis': axis, 'system': NAMES[key], 'cases': len(scores), 'mean': scores[key].mean(), 'ci_low': ci[0], 'ci_high': ci[1]})
axes_table = export(pd.DataFrame(axis_rows), 'rubric_grouped')
fig, ax = plt.subplots(figsize=(8, 4))
axes_table.pivot(index='axis', columns='system', values='mean').plot.barh(ax=ax)
ax.set_xlabel('Within-axis rubric score')
ax.set_ylabel('')
ax.set_yticklabels([label.get_text().replace('_quality', '').replace('_', ' ').capitalize() for label in ax.get_yticklabels()])
save_plot(fig, 'rubric_grouped')
ge = pd.read_csv(ROOT / 'evaluations/geval_clinical_protocols/geval_scores_framework_only.csv', sep=None, engine='python', encoding='utf-8-sig')
geval_rows = []
for (axis, key), group in ge.groupby(['metric', 'arm']):
    values = pd.to_numeric(group.score, errors='coerce').dropna()
    ci = boot_ci(values.to_numpy())
    label = 'D-1G' if key == 'delphi' else NAMES.get(key.removeprefix('baseline_'), key)
    geval_rows.append({'axis': axis, 'system': label, 'n': len(values), 'mean': values.mean(), 'ci_low': ci[0], 'ci_high': ci[1], 'at_least_4_5_pct': 100 * values.ge(4.5).mean()})
geval = export(pd.DataFrame(geval_rows), 'geval_grouped')
fig, ax = plt.subplots(figsize=(8, 4))
geval.pivot(index='axis', columns='system', values='mean').plot.barh(ax=ax)
ax.set_xlabel('Probability-weighted G-Eval score')
ax.set_ylabel('')
save_plot(fig, 'geval_grouped')
im = item_matrix(arms, ['P-base', *SINGLES])
im['members_met'] = im[['gpt', 'qwen', 'deepseekv4']].sum(axis=1)
recovery = []
for number, group in im.groupby('members_met'):
    count = int(group['P-base'].sum())
    ci = wilson(count, len(group))
    recovery.append({'members_met': int(number), 'items': len(group), 'panel_met': count, 'panel_met_fraction': count / len(group), 'ci_low': ci[0], 'ci_high': ci[1]})
recovery = export(pd.DataFrame(recovery), 'content_recovery')
missed = im[im[SINGLES].sum(axis=1).eq(0)]
export(pd.DataFrame([{'missed_by': 'all_seven_singles', 'items': len(missed), 'panel_met': int(missed['P-base'].sum()), 'panel_met_fraction': missed['P-base'].mean()}]), 'content_recovery_all_singles')
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.bar(recovery.members_met, recovery.panel_met_fraction, color='#2a78d6')
ax.errorbar(recovery.members_met, recovery.panel_met_fraction, yerr=[recovery.panel_met_fraction - recovery.ci_low, recovery.ci_high - recovery.panel_met_fraction], fmt='none', color='black', capsize=3)
ax.set_xticks(recovery.members_met, [str(x) + ' / 3' for x in recovery.members_met])
ax.set_xlabel('Independent members meeting the positive criterion')
ax.set_ylabel('Fraction met by D-1G')
save_plot(fig, 'content_recovery')
token_data = previous('clinical_system_scores.csv')
token_data = token_data[token_data.key.isin(['gpt', 'qwen', 'deepseekv4', 'P-base'])].copy()
export(token_data[['key', 'tokens_total_mean', 'tokens_completion_mean', 'median_words', 'judge_mean']], 'Fig3_tokens')
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh([NAMES[k] for k in token_data.key], token_data.tokens_total_mean / 1000, color=['#2a78d6' if k == 'P-base' else '#9a9a96' for k in token_data.key])
ax.set_xlabel('Recorded generation tokens per case (thousands)')
save_plot(fig, 'Fig3_tokens')
retests = repeated.groupby(['system', 'run']).y.mean().unstack('system')
fig, ax = plt.subplots(figsize=(7, 3.5))
retests.rename(columns=NAMES).plot(ax=ax, marker='o')
ax.set_ylabel('Mean rubric score')
save_plot(fig, 'retests_grouped')
h = pd.read_csv(human / 'protocol_overall.csv')
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.errorbar(h['mean'], np.arange(len(h)), xerr=[h['mean'] - h.ci_low, h.ci_high - h['mean']], fmt='o', color='#2a78d6')
ax.set_yticks(np.arange(len(h)), h.source)
ax.set_xlabel('Mean human rubric score')
save_plot(fig, 'human_systems_grouped')
hax = pd.read_csv(human / 'protocol_axis_means.csv')
fig, ax = plt.subplots(figsize=(8, 4))
hax.pivot(index='axis', columns='source', values='mean').plot.barh(ax=ax)
ax.set_xlabel('Mean human axis score')
save_plot(fig, 'human_axes_grouped')
human_agreement = pd.read_csv(human / 'human_llm.csv')
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.scatter(human_agreement.kappa, np.arange(len(human_agreement)), color='#2a78d6')
ax.set_yticks(np.arange(len(human_agreement)), human_agreement.a + ' / ' + human_agreement.b)
ax.set_xlabel('Criterion-level Cohen kappa')
save_plot(fig, 'human_validation')
display(pd.DataFrame({'output': list(exports), 'rows': [len(x) for x in exports.values()]}))


,output,rows
0,baselines_matrix,26
1,panels_matrix,16
2,score_distribution,18
3,variance,6
4,source_difficulty,3
5,source_difficulty_model,2
6,subgroup_tests,4
7,subgroups,32
8,length_regression,18
9,organizer_regression,3


In [10]:
display(pd.DataFrame({'table': list(exports), 'rows': [len(frame) for frame in exports.values()]}))


,table,rows
0,baselines_matrix,26
1,panels_matrix,16
2,score_distribution,18
3,variance,6
4,source_difficulty,3
5,source_difficulty_model,2
6,subgroup_tests,4
7,subgroups,32
8,length_regression,18
9,organizer_regression,3
